## Prepare questions from Brainlat for training GRPO

Created by: Sahana Kowshik

Date: 11/2025

In [1]:
import pandas as pd
import json
import random

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/brainlat"
test = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/test_summary.csv")
test = test[(test['Diagnosis'] != "Multiple sclerosis") & (test['Diagnosis'] != "Parkinson's disease")].sample(frac=1, random_state=42).reset_index(drop=True)

In [3]:
test.iloc[1]['visit_summary']

'The patient is a 69-year-old right-handed male with 18 years of education. On the Montreal Cognitive Assessment (MoCa), the total score was 18 out of 30, indicating cognitive impairment. The visuospatial assessment score was 2 out of 5, the recognition assessment score was 2 out of 3, the attention assessment score was 5 out of 6, the language assessment score was 3 out of 3, the abstraction assessment score was 2 out of 2, the memory assessment score was 0 out of 5, and the orientation assessment score was 4 out of 6. \n\nImaging evidence was derived from T1-weighted MRI scans analyzed using a Generalized Additive Model for Location, Scale, and Shape (GAMLSS), which was fitted to FastSurfer-derived regional brain volumes from 2,087 healthy controls. This method followed the Brain charts for the human lifespan methodology published in Nature in 2022. The data were aggregated from five independent cohorts, and the analysis included region-of-interest (ROI) volumes for the medial and la

In [4]:
len(test)

660

In [5]:
test['Diagnosis'].value_counts()

Diagnosis
Alzheimer's disease                           269
Healthy                                       242
Behavioral variant frontotemporal dementia    149
Name: count, dtype: int64

In [6]:
columns = ['ID', 'patient_summary', 'diag_summary', 'visit_summary', 'question', 'options', 'ground_truth', 'ground_truth_text', 'COHORT']

In [7]:
test['diag_summary'] = test['Diagnosis']
test["COHORT"] = "Brainlat"

In [8]:
test.columns

Index(['Diagnosis', 'Metadata', 'EEG Report', 'MRI Report', 'patient_summary',
       'ID', 'visit_summary_prompt', 'visit_summary', 'diag_summary',
       'COHORT'],
      dtype='object')

In [9]:
print(json.dumps(json.loads(test.iloc[500]['Metadata']), indent=4))

{
    "Sex": "Female",
    "Age": 73.0,
    "Years of education": 17.0,
    "Total score of INECO Frontal Screening (0\u201330)": 21.0,
    "Motor Programming INECO Frontal Screening subset (0\u20133)": 1.0,
    "Conflicting Instructions INECO Frontal Screening subset (0\u20133)": 3.0,
    "Go/No Go INECO Frontal Screening subset (0\u20133)": 3.0,
    "Backward Digit Span INECO Frontal Screening subset (0\u20136)": 4.0,
    "Verbal Working Memory INECO Frontal Screening subset (0\u20132)": 2.0,
    "Spatial Working Memory INECO Frontal Screening subset (0\u20134)": 2.0,
    "Abstraction Capacity INECO Frontal Screening subset (0\u20133)": 2.0,
    "Verbal Inhibitory Control INECO Frontal Screening subset (0\u20136)": 4.0,
    "Reduced facial emotion recognition test (FERT), a mini-SEA component (0\u201335)": 10.7,
    "Shortened version of the Faux-Pas test (FPT), a mini-SEA component (0\u201340)": 10.5
}


## Cognitive status

In [10]:
import random, string

# Question variants for cognitive status identification
cog_question_variants = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

# # Original answer texts
# COG_OPTION_TEXTS = [
#     "Normal Cognition (NC)",
#     "Cognitively Impaired (IMP)",
# ]

# Original answer texts
COG_OPTION_TEXTS = [
    "Normal Cognition (NC)",
    "Mild Cognitive Impairment (MCI)",
    "Dementia (DE)"
]

# Mapping from NACCUDSD to the correct label text
COG_ANSWER_MAP = {
    "Alzheimer's disease": COG_OPTION_TEXTS[2],
    # "Parkinson's disease": COG_OPTION_TEXTS[1],
    'Behavioral variant frontotemporal dementia': COG_OPTION_TEXTS[2],
    'Healthy': COG_OPTION_TEXTS[0],
}

def add_cog_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Random question phrasing
    row['question'] = rng.choice(cog_question_variants)

    # 2️⃣ Shuffle the answer options
    shuffled = COG_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Determine correct answer letter
    correct_text = COG_ANSWER_MAP.get(row['Diagnosis'], None)
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # fallback for unrecognized codes
    
    row['ground_truth_text'] = correct_text

    return row


In [11]:
test_cog = test[test['Diagnosis'] != "Parkinson's disease"].reset_index(drop=True)
# Apply the function to the MCI training set
test_cog = test_cog.apply(add_cog_question, axis=1)

In [12]:
test_cog['ground_truth_text'].value_counts()

ground_truth_text
Dementia (DE)            418
Normal Cognition (NC)    242
Name: count, dtype: int64

In [13]:
print(test_cog.iloc[10]['ground_truth'])
print(test_cog.iloc[10]['ground_truth_text'])
print(test_cog.iloc[10]['Diagnosis'])
print(test_cog.iloc[10]['options'])

C
Normal Cognition (NC)
Healthy
A. Dementia (DE)
B. Mild Cognitive Impairment (MCI)
C. Normal Cognition (NC)


In [14]:
test_cog = test_cog[columns]
print(len(test_cog))
test_cog = test_cog.dropna(how='any').reset_index(drop=True)
print(len(test_cog))

660
660


In [15]:
test_cog['Q_TYPE'] = "Cognitive status"

In [16]:
test_cog.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,sub-CLB00196,"{""Metadata"": {""Sex"": ""Female"", ""Age"": 77.0, ""Y...",Alzheimer's disease,The patient is a 77-year-old female with 12 ye...,"From the information available, determine the ...",A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,A,Dementia (DE),Brainlat,Cognitive status
1,sub-COB128,"{""Metadata"": {""Sex"": ""Male"", ""Age"": 69.0, ""Yea...",Behavioral variant frontotemporal dementia,The patient is a 69-year-old right-handed male...,Identify the patient's cognitive status based ...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,C,Dementia (DE),Brainlat,Cognitive status
2,sub-CLB00182,"{""Metadata"": {""Sex"": ""Female"", ""Age"": 72.0, ""Y...",Healthy,The patient is a 72-year-old female with 18 ye...,"Using the information provided, determine the ...",A. Mild Cognitive Impairment (MCI)\nB. Dementi...,C,Normal Cognition (NC),Brainlat,Cognitive status
3,sub-MXB00029,"{""Metadata"": {""Sex"": ""Male"", ""Age"": 77.0, ""Yea...",Alzheimer's disease,The patient is a 77-year-old male with nine ye...,Identify the patient's cognitive status based ...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,C,Dementia (DE),Brainlat,Cognitive status
4,sub-COA00012,"{""Metadata"": {""Sex"": ""Male"", ""Age"": 62.0, ""Yea...",Healthy,The patient is a 62-year-old right-handed male...,Identify the patient's cognitive status based ...,A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,B,Normal Cognition (NC),Brainlat,Cognitive status


In [17]:
test_cog.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_cog.csv", index=False)

## Primary etiology

In [18]:
import random, string

etiology_question_variants = [
    "Identify the primary etiologic diagnosis of the patient using the information provided, if applicable.",
    "Identify the primary etiology of the patient's cognitive impairment using the information provided, if applicable.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment, if applicable.",
    "Using the information provided, identify the primary cause of the patient's cognitive decline, if applicable.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information, if applicable."
]

# Raw etiology answer texts in original order
ETIOLOGY_OPTION_TEXTS = [
    "Alzheimer's disease (AD)",
    "Lewy body disease (LBD)",
    "Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)",
    "Vascular brain injury or vascular dementia including stroke (VD)",
    "Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)",
    "Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)",
    "Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)",
    "Not applicable (no cognitive impairment)",
]

# Mapping of ETPR code to correct answer text
ETPR_ANSWER_MAP = {
    "Alzheimer's disease": ETIOLOGY_OPTION_TEXTS[0],
    # "Parkinson's disease": ETIOLOGY_OPTION_TEXTS[1],
    'Behavioral variant frontotemporal dementia': ETIOLOGY_OPTION_TEXTS[2],
    'Healthy': ETIOLOGY_OPTION_TEXTS[-1],
}

def add_etpr_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly select a question variant
    row['question'] = rng.choice(etiology_question_variants)

    # 2️⃣ Shuffle options and assign fresh letters
    shuffled = ETIOLOGY_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Get correct text from ETPR code (default to "Not applicable" if unknown)
    correct_text = ETPR_ANSWER_MAP.get(row['Diagnosis'])

    if correct_text is None:
        raise ValueError(f"Invalid ETPR value: {row['Diagnosis']}")


    # 4️⃣ Get ground truth letter from shuffled list
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # shouldn't happen unless text mismatch
        
    row['ground_truth_text'] = correct_text

    return row


In [19]:
# Apply the function to the MCI training set
test_etpr = test.apply(add_etpr_question, axis=1)

In [20]:
test_etpr[test_etpr["ground_truth_text"] == "Not applicable (no cognitive impairment)"]['Diagnosis'].value_counts()

Diagnosis
Healthy    242
Name: count, dtype: int64

In [21]:
print(test_etpr.iloc[400]['ground_truth'])
print(test_etpr.iloc[400]['ground_truth_text'])
print(test_etpr.iloc[400]['Diagnosis'])
print(test_etpr.iloc[400]['options'])

F
Alzheimer's disease (AD)
Alzheimer's disease
A. Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)
B. Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)
C. Lewy body disease (LBD)
D. Vascular brain injury or vascular dementia including stroke (VD)
E. Not applicable (no cognitive impairment)
F. Alzheimer's disease (AD)
G. Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)
H. Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)


In [22]:
test_etpr = test_etpr[columns]
test_etpr['Q_TYPE'] = "Primary etiology"

In [23]:
print(test_etpr.iloc[-1]['diag_summary'])

Behavioral variant frontotemporal dementia


In [24]:
test_etpr['ground_truth_text'].value_counts()

ground_truth_text
Alzheimer's disease (AD)                                                                                                                                                                                             269
Not applicable (no cognitive impairment)                                                                                                                                                                             242
Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)    149
Name: count, dtype: int64

In [25]:
test_etpr.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,sub-CLB00196,"{""Metadata"": {""Sex"": ""Female"", ""Age"": 77.0, ""Y...",Alzheimer's disease,The patient is a 77-year-old female with 12 ye...,"Using the information provided, identify the p...",A. Not applicable (no cognitive impairment)\nB...,G,Alzheimer's disease (AD),Brainlat,Primary etiology
1,sub-COB128,"{""Metadata"": {""Sex"": ""Male"", ""Age"": 69.0, ""Yea...",Behavioral variant frontotemporal dementia,The patient is a 69-year-old right-handed male...,Identify the primary etiology of the patient's...,A. Psychiatric conditions including schizophre...,G,Frontotemporal lobar degeneration and its vari...,Brainlat,Primary etiology
2,sub-CLB00182,"{""Metadata"": {""Sex"": ""Female"", ""Age"": 72.0, ""Y...",Healthy,The patient is a 72-year-old female with 18 ye...,Identify the primary etiologic diagnosis of th...,A. Vascular brain injury or vascular dementia ...,E,Not applicable (no cognitive impairment),Brainlat,Primary etiology
3,sub-MXB00029,"{""Metadata"": {""Sex"": ""Male"", ""Age"": 77.0, ""Yea...",Alzheimer's disease,The patient is a 77-year-old male with nine ye...,Identify the primary etiology of the patient's...,A. Lewy body disease (LBD)\nB. Psychiatric con...,D,Alzheimer's disease (AD),Brainlat,Primary etiology
4,sub-COA00012,"{""Metadata"": {""Sex"": ""Male"", ""Age"": 62.0, ""Yea...",Healthy,The patient is a 62-year-old right-handed male...,Identify the primary etiology of the patient's...,A. Lewy body disease (LBD)\nB. Frontotemporal ...,D,Not applicable (no cognitive impairment),Brainlat,Primary etiology


In [26]:
test_etpr["ground_truth"].value_counts()

ground_truth
D    97
C    94
H    86
B    80
A    78
G    77
F    74
E    74
Name: count, dtype: int64

In [27]:
test_etpr.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_etpr.csv", index=False)